# DAY 4 - Airbnb listing with view from balconies in Budapest, Hungary

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager
import mapclassify
import geopandas as gpd
import requests

### Load Data

In [2]:
listings = pd.read_csv('listings.csv.gz', low_memory=False)

In [3]:
reviews1 = pd.read_csv('reviews.csv.gz', compression='gzip')

In [35]:
from shapely.geometry import Point

listings['geometry'] = listings.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)
listing_df = gpd.GeoDataFrame(listings, geometry='geometry',crs='EPSG:4326')

In [37]:
#Sanity Check
listing_df = listing_df[['id', 'geometry']]
listing_df.head(5)

,id,geometry
0,6020,POINT (19.04829 47.51425)
1,6022,POINT (19.04871 47.51481)
2,15309,POINT (19.05455 47.50402)
3,15313,POINT (19.06123 47.49479)
4,15315,POINT (19.06771 47.50432)


In [83]:
reviews1 = reviews1[["id", "comments"]]
reviews1.head(2)

,id,comments
0,111984,Janos was a terrific host! He spent a lot of ...
1,175561,Janos and Jozef were great hosts. Not only wa...


### Check Duplicates

In [15]:
duplicate = reviews1[reviews1.duplicated(subset=['id'], keep=False)]
duplicate_count = reviews1.duplicated(subset=['id']).sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 105


In [17]:
duplicates = duplicate.sort_values(by='id')
print(duplicates)

                  listing_id                   id        date  reviewer_id  \
1146173  1613027629019737051  1297768286539383851  2024-11-25    452552867   
1030502  1196890139538390728  1297768286539383851  2024-11-25    452552867   
1146174  1613027629019737051  1325325345157288778  2025-01-02    596860956   
1030584  1196890139538390728  1325325345157288778  2025-01-02    596860956   
1146175  1613027629019737051  1326763226363526944  2025-01-04    453758981   
...                      ...                  ...         ...          ...   
1031108  1196890139538390728  1773725173700538152  2026-09-13    539369353   
1031109  1196890139538390728  1773773733294335959  2026-09-13    216533318   
1146222  1613027629019737051  1773773733294335959  2026-09-13    216533318   
1151062  1741889366646134518  1777451114805565720  2026-09-18    412887119   
1151067  1741896628534981935  1777451114805565720  2026-09-18    412887119   

        reviewer_name                                          

In [39]:
duplicates =listing_df[listing_df.duplicated(subset=['id'], keep=False)]
duplicate_count = listing_df.duplicated(subset=['id']).sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 0


In [41]:
#Remove duplicates
df_cleaned = reviews1.drop_duplicates(subset=['id'], keep='first')

In [43]:
listing_df = listing_df.to_crs('EPSG:23700')

In [45]:
len(listing_df)

11240

In [47]:
len(reviews1)

1151191

### Join the listings and reviews

In [49]:
merged_df = listing_df.merge(df_cleaned, left_on='id', right_on='listing_id', how='inner')

In [51]:
merged_df.head(1)

,id_x,geometry,listing_id,id_y,date,reviewer_id,reviewer_name,comments
0,6020,POINT (650063.687 241147.099),6020,7941,2009-08-21,18127,Flannery,Janos was very patient with us as we were late...


## First FIlter : View

In [53]:
import re

# Define first regex expression
core_view_regex = (
    r'(?i)\b('
    r'kilátás(?:ban|sal|ú|é)?|panorám[aá](?:s|val|ból)?|látványos|kilátó|'
    r'view|vista|scenic|panorama|panoramic|lookout|overlook|scenery|picturesque'
    r')\b'
)


mask_1 = (merged_df['comments'].str.contains(core_view_regex, na=False, regex=True))
result_1 = merged_df[mask_1].copy()
print(f"Result 1 (Core view keywords): {len(result_1)} rows")

C:\Users\nooriza maharani\AppData\Local\Temp\ipykernel_13636\2891949317.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_1 = (merged_df['comments'].str.contains(core_view_regex, na=False, regex=True))


Result 1 (Core view keywords): 20432 rows


In [57]:
result_1.head(1)

,id_x,geometry,listing_id,id_y,date,reviewer_id,reviewer_name,comments
23,6020,POINT (650063.687 241147.099),6020,2972183,2012-11-26,1271028,Katalin,Janos and Jozsef are really friendly and welco...


## Second Filter : Descriptive View

In [61]:
import re

descriptive_regex = (
    r'(?i)\b('
    r'csodás|csodálatos|lenyűgöző|bámulatos|lélegzetelállító|gyönyörű|pazar|pompás|'
    r'gorgeous|breathtaking|stunning|wonderful|amazing|beautiful'
    r')\b'
)

mask_2 = (result_1['comments'].str.contains(descriptive_regex, na=False, regex=True))
result_2 = result_1[mask_2].copy()
print(f"Result 2 (Core view keywords): {len(result_2)} rows")

C:\Users\nooriza maharani\AppData\Local\Temp\ipykernel_13636\739744918.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_2 = (result_1['comments'].str.contains(descriptive_regex, na=False, regex=True))


Result 2 (Core view keywords): 10381 rows


In [63]:
result_2.head(1)

,id_x,geometry,listing_id,id_y,date,reviewer_id,reviewer_name,comments
32,6020,POINT (650063.687 241147.099),6020,4830270,2013-05-28,495511,Jasna,"The flat is well located, in a beautiful old b..."


In [67]:
# Allow full text to be displayed in columns
pd.set_option('display.max_colwidth', None)

# Now when you display your dataframe, the comments won't be cut off
result_2[['listing_id', 'comments']].head()

,listing_id,comments
32,6020,"The flat is well located, in a beautiful old building. Danube can be seen though only if you position yourself well out of the window turning right. Don't expect the Danube view from inside the flat. All in all, everything looks much better on the photos than in reality. Many things are just worn out and could do with renovation. You don't see those on the pictures. Mattress is the worse one we ever slept on. Of course we don't expect the extra class mattress when travelling but we both returned with strong back pain. However, the worse part of our stay was Janos attitutude of making small, additional earnings with his guests. After the booking we were told that the internet will additionally cost 1,50 per 24 hours. On arrival, it was clear that Janos calculates 1,50 per 12 hours. The fact that there was no connection for the first 24 hrs did not make him eventually offer a refund. He told us instead that we were supposed to ring him from our mobile so that he can fix it. In the flat itself there is a long, unplesant list of 'house rules' and you can also buy items like a jar of marmelade from 2011 (staying out of the fridge) for 7 € (!) or washing liquid for 1 €. At the end of our stay, Janos even tried to sell us a taxi ride to the airport explaining how the mini bus is more expensive (false, by the way). Budapest was beautiful as always but this was our worse airbnb experience. There are things in life, Janos, that are more precious than 1 € you try to make on your guests!"
157,6022,"Details for receiving the key were fabulous and all went smoothly. I have to say that the description is a bit inaccurate. There is no Danube view and one of the bedrooms is also the lounge/dining area. I did ask Janos if it was the same apartment as shown in the Airbnb listing. The listing just didn't depict the layout. In saying that, the best aspect were the hosts. The apartment was clean and nicely made up, and both Janos and Joszef came and gave us perfect instructions to settle in, helped with the luggage and went through ideas of what to see. During our stay I developed a toothache and Janos was amazing. He found me a dentist and even offered to come with me to hold my hand in case I got lost or scared. (so sweet). He checked to see if I got medication for the pain and was so lovely and helpful. The guys are absolutely top notch hosts and work really hard. The apartment is in a lovely old building; we met some of the neighbours. The street that it sits on has lots of restaurants - a McDonald with free wi-fi about a minute's walk away - and is close to public transport. We did a lot of walking in the area and really got the feeling of Budapest. The Parliament building (10 minutes walk) would have to be the most beautiful building in the world (and I have travelled a lot). Take time just to sit on a seat in the park near the entrance and watch as day turns into dusk and the lights start coming on. I would stay at this apartment again and look forward to those renovations guys. Many thanks."
169,6022,"We don’t want to say to many nice things about this apartment for the simple reason People will want to book it and we can’t get back on the dates we want. But in all honesty to sum our stay up in a word. OUTSTANDING. We have stayed in many Airbnb apartments around the world and can honestly say this is the best one by far. If we could rate this more than 5 star we would. The pictures and description are 100% accurate. Classy and has lots of style, it is spotlessly clean, Very comfortable a home from home. Perfectly located by the Duna river, excellent for everything, scenery, restaurants, walks, transport and shopping. This apartment has everything you need and more, we did not know the hosts Jozsef and Janos before we stayed but very glad we met them, we can honestly say they are two exceptional individuals that have pride in their apartment and really know how to look after guest. My wife and I will be back for

### Filter 3 : Balcony

In [72]:
balcony_regex = r"(?i)\b(erkély|erkélyek|erkélyes|balcony|balconies)\b"

mask_3 = (result_2['comments'].str.contains(balcony_regex, na=False, regex=True))
result_3 = result_2[mask_3].copy()
print(f"Result 3 (Core view keywords): {len(result_3)} rows")

C:\Users\nooriza maharani\AppData\Local\Temp\ipykernel_13636\4223218523.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_3 = (result_2['comments'].str.contains(balcony_regex, na=False, regex=True))


Result 3 (Core view keywords): 2091 rows


## Viz

In [82]:
result_3.to_file("output_data.gpkg", layer="my_spatial_layer", driver="GPKG")

## Disclaimer

The filter applied still returns comments such as :  

'There's *no* visible Danube *View* on the *balcony*"
or
"I couldn't see any *wonderful* *scenery* from the *balcony*"

I am still exploring ways on how to isolate those type of reviews